# Imports and Setup

In [39]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.callbacks import EarlyStopping

print("All imports successful")
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("Num CPUs Available:", len(tf.config.list_physical_devices('CPU')))

All imports successful
TensorFlow version: 2.20.0
Num GPUs Available: 0
Num CPUs Available: 1


In [40]:
print("TensorFlow version:", tf.__version__)

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

print("Num CPUs Available:", len(tf.config.list_physical_devices('CPU')))

TensorFlow version: 2.20.0
Num GPUs Available: 0
Num CPUs Available: 1


# Data Loading and Initial Setup

In [41]:
df = pd.read_csv("../data/metadata")
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern


In [42]:
image_dir = "../data/images"

df["image_path"] = df["image_id"].apply(
    lambda x: os.path.join(image_dir, x + ".jpg")
)

df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern,../data/images/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern,../data/images/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern,../data/images/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern,../data/images/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern,../data/images/ISIC_0031633.jpg


In [43]:
print("Missing images:", df["image_path"].apply(lambda x: not os.path.exists(x)).sum())

Missing images: 0


In [44]:
df.columns

Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset', 'image_path'],
      dtype='object')

In [45]:
df["dx"].value_counts()

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

# Data Cleaning and Preprocessing

In [46]:
df_model = df[['image_path', 'dx', 'age', 'localization']].copy()
df_model.head()

,image_path,dx,age,localization
0,../data/images/ISIC_0027419.jpg,bkl,80.0,scalp
1,../data/images/ISIC_0025030.jpg,bkl,80.0,scalp
2,../data/images/ISIC_0026769.jpg,bkl,80.0,scalp
3,../data/images/ISIC_0025661.jpg,bkl,80.0,scalp
4,../data/images/ISIC_0031633.jpg,bkl,75.0,ear


In [47]:
df_model.isnull().sum()

image_path       0
dx               0
age             57
localization     0
dtype: int64

In [48]:
df_model["age"] = df["age"].fillna(df["age"].median())

In [49]:
df_model = pd.get_dummies(df_model, columns=['localization'])

le = LabelEncoder()
df_model['dx'] = le.fit_transform(df_model['dx'])

In [50]:
for i, class_name in enumerate(le.classes_):
    print(i, ":", class_name)

0 : akiec
1 : bcc
2 : bkl
3 : df
4 : mel
5 : nv
6 : vasc


In [51]:
y = df_model["dx"]

X_meta = df_model.drop(columns=['image_path', 'dx'])

In [52]:
df_model.head()

,image_path,dx,age,localization_abdomen,localization_acral,localization_back,localization_chest,localization_ear,localization_face,localization_foot,localization_genital,localization_hand,localization_lower extremity,localization_neck,localization_scalp,localization_trunk,localization_unknown,localization_upper extremity
0,../data/images/ISIC_0027419.jpg,2,80.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
1,../data/images/ISIC_0025030.jpg,2,80.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
2,../data/images/ISIC_0026769.jpg,2,80.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
3,../data/images/ISIC_0025661.jpg,2,80.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
4,../data/images/ISIC_0031633.jpg,2,75.0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False


# Multimodal Classifier

## Stratified K-Fold Cross-Validation

In [53]:
y = df_model["dx"]
X_images = df_model["image_path"]
X_meta = df_model.drop(columns=['image_path', 'dx'])

print("Metadata shape:", X_meta.shape)
print("Images:", X_images.shape)
print("dx:", y.shape)

Metadata shape: (10015, 16)
Images: (10015,)
dx: (10015,)


In [54]:
bool_cols = X_meta.select_dtypes(include=["bool"]).columns
X_meta[bool_cols] = X_meta[bool_cols].astype(int)

In [55]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [56]:
folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta, y), start=1):
    fold_data = {
        "fold": fold,
        "X_meta_train": X_meta.iloc[train_idx].reset_index(drop=True),
        "X_meta_val": X_meta.iloc[val_idx].reset_index(drop=True),
        "X_img_train": X_images.iloc[train_idx].reset_index(drop=True),
        "X_img_val": X_images.iloc[val_idx].reset_index(drop=True),
        "y_train": y.iloc[train_idx].reset_index(drop=True),
        "y_val": y.iloc[val_idx].reset_index(drop=True)
    }
    folds.append(fold_data)
print("Number of folds saved:", len(folds))

Number of folds saved: 5


## Image Preprocessing and Feature Extraction

In [57]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

print("Model loaded")

Model loaded


In [21]:
"""
def extract_features(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((224, 224))

    img_array = np.array(img, dtype=np.float32)
    img_array = np.expand_dims(img_array, axis=0)

    img_array = preprocess_input(img_array)

    features = base_model.predict(img_array, verbose=0)
    return features[0]
"""

In [59]:
"""
def load_and_preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((224, 224))
    img_array = np.array(img, dtype=np.float32)
    return preprocess_input(img_array)
"""

def extract_features_batch(image_paths, batch_size=32):
    all_features = []
    
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch_images = np.array([load_and_preprocess_image(p) for p in batch_paths])
        
        batch_features = base_model.predict(batch_images, verbose=0)
        all_features.append(batch_features)
        
    return np.vstack(all_features)

## Multimodal Classifier and Evaluation

In [60]:
all_macro_f1 = []
all_reports = []
all_cm = []

class_names = le.classes_
num_classes = len(class_names)

label_map = {
    "akiec": "Actinic Keratosis",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevi",
    "vasc": "Vascular Lesion"
}

for i, fold in enumerate(folds):
    X_img_train = fold["X_img_train"].tolist()
    X_img_val = fold["X_img_val"].tolist()

    X_train_img_features = extract_features_batch(X_img_train, batch_size=16)
    X_val_img_features = extract_features_batch(X_img_val, batch_size=16)

    X_train_meta = fold["X_meta_train"].to_numpy(dtype=np.float32)
    X_val_meta = fold["X_meta_val"].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    X_train_meta_scaled = scaler.fit_transform(X_train_meta)
    X_val_meta_scaled = scaler.transform(X_val_meta)


    X_train_combined = np.concatenate([X_train_img_features, X_train_meta_scaled], axis=1)
    X_val_combined = np.concatenate([X_val_img_features, X_val_meta_scaled], axis=1)


    y_train = fold["y_train"].to_numpy()
    y_val = fold["y_val"].to_numpy()

    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = dict(zip(classes, weights))

    classifier = keras.Sequential([
        keras.Input(shape=(X_train_combined.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])

    classifier.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = classifier.fit(
        X_train_combined,
        y_train,
        validation_data=(X_val_combined, y_val),
        epochs=50,
        batch_size=32,
        verbose=0,
        class_weight=class_weights,
        callbacks=[early_stop]
    )

    y_pred = classifier.predict(X_val_combined)
    y_pred_classes = np.argmax(y_pred, axis=1)

    report = classification_report(
        y_val,
        y_pred_classes,
        target_names=[label_map[name] for name in class_names]
    )

    macro_f1 = f1_score(y_val, y_pred_classes, average="macro")
    all_macro_f1.append(macro_f1)

    fold_report = f"FOLD {i+1}\n{report}\nMacro F1: {macro_f1:.3f}"
    all_reports.append(fold_report)

    cm = confusion_matrix(y_val, y_pred_classes)
    all_cm.append(cm)

# SUMMARY REPORT
mean_macro_f1 = np.mean(all_macro_f1)

summary_text = "\n\n".join(all_reports)
summary_text += f"\n\n{'=' * 90}\nMEAN MACRO F1 ACROSS 5 FOLDS: {mean_macro_f1:.3f}"

plt.figure(figsize=(14, 30))
plt.text(
    0.01,
    0.99,
    summary_text,
    fontsize=11,
    family="monospace",
    va="top"
)
plt.axis("off")

plt.savefig(
    "../outputs/classification_reports/mobilenet_multimodal_5fold_summary.png",
    bbox_inches="tight",
    pad_inches=0.4,
    dpi=300
)

plt.close()

# CONFUSION MATRIX
mean_cm = np.mean(all_cm, axis=0)

plt.figure(figsize=(8, 6))
plt.imshow(mean_cm)
plt.title("Average Confusion Matrix (MobileNetV2 Multimodal Classifier, 5 Folds)")
plt.colorbar()
plt.xticks(
    range(len(class_names)),
    [label_map[c] for c in class_names],
    rotation=90
)
plt.yticks(
    range(len(class_names)),
    [label_map[c] for c in class_names]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()

plt.savefig(
    "../outputs/confusion_matrices/mobilenet_multimodal_confusion_matrix_5fold_average.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 778us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 724us/step


# Image-Only Classifier

## Stratified K-Fold Cross-Validation

In [61]:
y = df_model["dx"]
X_images = df_model["image_path"]

print("Images:", X_images.shape)
print("dx:", y.shape)

Images: (10015,)
dx: (10015,)


In [63]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [64]:
folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_images, y), start=1):
    fold_data = {
        "fold": fold,
        "X_img_train": X_images.iloc[train_idx].reset_index(drop=True),
        "X_img_val": X_images.iloc[val_idx].reset_index(drop=True),
        "y_train": y.iloc[train_idx].reset_index(drop=True),
        "y_val": y.iloc[val_idx].reset_index(drop=True)
    }
    folds.append(fold_data)
print("Number of folds saved:", len(folds))

Number of folds saved: 5


## Image Preprocessing and Feature Extraction

In [65]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

print("Model loaded")

Model loaded


In [31]:
"""
def extract_features(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((224, 224))

    img_array = np.array(img, dtype=np.float32)
    img_array = np.expand_dims(img_array, axis=0)

    img_array = preprocess_input(img_array)

    features = base_model.predict(img_array, verbose=0)
    return features[0] 
"""

In [66]:
"""
def load_and_preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((224, 224))
    img_array = np.array(img, dtype=np.float32)
    return preprocess_input(img_array)
"""

def extract_features_batch(image_paths, batch_size=32):
    all_features = []
    
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch_images = np.array([load_and_preprocess_image(p) for p in batch_paths])
        
        batch_features = base_model.predict(batch_images, verbose=0)
        all_features.append(batch_features)
        
    
    return np.vstack(all_features)

## Image-Only Classifier and Evaluation

In [34]:
all_macro_f1 = []
all_reports = []
all_cm = []

class_names = le.classes_

label_map = {
    "akiec": "Actinic Keratosis",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevi",
    "vasc": "Vascular Lesion"
}

for i, fold in enumerate(folds):

    X_img_train = fold["X_img_train"].tolist()
    X_img_val = fold["X_img_val"].tolist()

    X_train_img_features = extract_features_batch(X_img_train, batch_size=16)
    X_val_img_features = extract_features_batch(X_img_val, batch_size=16)

    y_train = fold["y_train"].to_numpy()
    y_val = fold["y_val"].to_numpy()

    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = dict(zip(classes, weights))

    classifier = keras.Sequential([
        keras.Input(shape=(X_train_img_features.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(len(class_names), activation="softmax")
    ])

    classifier.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = classifier.fit(
        X_train_img_features,
        y_train,
        validation_data=(X_val_img_features, y_val),
        epochs=50,
        batch_size=32,
        verbose=0,
        class_weight=class_weights,
        callbacks=[early_stop]
    )

    y_pred = classifier.predict(X_val_img_features)
    y_pred_classes = np.argmax(y_pred, axis=1)

    report = classification_report(
        y_val,
        y_pred_classes,
        target_names=[label_map[name] for name in class_names]
    )

    macro_f1 = f1_score(y_val, y_pred_classes, average="macro")
    all_macro_f1.append(macro_f1)

    fold_report = f"FOLD {i+1}\n{report}\nMacro F1: {macro_f1:.3f}"
    all_reports.append(fold_report)

    cm = confusion_matrix(y_val, y_pred_classes)
    all_cm.append(cm)

# 1) SUMMARY REPORT
mean_macro_f1 = np.mean(all_macro_f1)

summary_text = "\n\n".join(all_reports)
summary_text += f"\n\n{'=' * 90}\nMEAN MACRO F1 ACROSS 5 FOLDS: {mean_macro_f1:.3f}"

plt.figure(figsize=(14, 30))
plt.text(
    0.01,
    0.99,
    summary_text,
    fontsize=11,
    family="monospace",
    va="top"
)
plt.axis("off")

plt.savefig(
    "../outputs/classification_reports/mobilenet_image_only_5fold_summary.png",
    bbox_inches="tight",
    pad_inches=0.4,
    dpi=300
)

plt.close()

# 2) CONFUSION MATRIX
mean_cm = np.mean(all_cm, axis=0)

plt.figure(figsize=(8, 6))
plt.imshow(mean_cm)
plt.title("Average Confusion Matrix (MobileNetV2 Image-Only Classifier, 5 Folds)")
plt.colorbar()
plt.xticks(
    range(len(class_names)),
    [label_map[name] for name in class_names],
    rotation=90
)
plt.yticks(
    range(len(class_names)),
    [label_map[name] for name in class_names]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()

plt.savefig(
    "../outputs/confusion_matrices/mobilenet_image_only_confusion_matrix_5fold_average.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 893us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 789us/step


KeyboardInterrupt: 